# M3.1 · Skaler autograd (micrograd)

PyTorch'un kalbindeki mekanizmayı kendin kuruyorsun. Karpathy'nin klasik örneğinde geri geçişi elle yapıyor, saf Python'da `Value` sınıfıyla bir autograd motoru yazıyor, aynı nöronu NumPy'da batch hâlinde elle türetiyor, PyTorch'la birebir karşılaştırıyor ve en sonda örnek başına gradyanları `vmap` ile döngüsüz hesaplıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m3-1-micrograd.html](https://m-gonul.github.io/pytorch-lab/m3-1-micrograd.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m3_1_micrograd.py
CHECKS = {
 "b": "import math\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    R.test(\"`Value` sınıfı tanımlı\", lambda: need(ns, \"Value\", \"sınıf\"))\n    V = ns.get(\"Value\")\n    if V is None:\n        return\n\n    def hand():\n        a, b, c, f = V(2), V(-3), V(10), V(-2)\n        L = (a * b + c) * f\n        L.backward()\n        close([L.data, a.grad, b.grad, c.grad, f.grad], [-8, 6, -4, -2, 4])\n    R.test(\"elle örnek: L = −8; ∂a = 6, ∂b = −4, ∂c = −2, ∂f = 4\", hand)\n\n    def neuron():\n        x1, x2, w1, w2, b = V(2), V(0), V(-3), V(1), V(6.8813735870195432)\n        o = (x1 * w1 + x2 * w2 + b).tanh()\n        o.backward()\n        close([o.data, w1.grad, w2.grad, x1.grad, x2.grad], [2 ** -0.5, 1.0, 0.0, -1.5, 0.5], atol=1e-6)\n    R.test(\"tanh nöronu: o = 1/√2, ∂w₁ = 1, ∂x₁ = −1.5\", neuron)\n\n    def reuse():\n        a = V(3.0)\n        b = a + a\n        b.backward()\n        ensure(abs(a.grad - 2) < 1e-12, f\"a + a için ∂/∂a = {a.grad}, 2 bekleniyordu (+= yerine = mi yazdın?)\")\n        a2 = V(-4.0)\n        c = a2 * a2\n        c.backward()\n        ensure(abs(a2.grad + 8) < 1e-12, f\"a·a için ∂/∂a = {a2.grad}, 2a = −8 bekleniyordu\")\n    R.test(\"paylaşılan düğümde gradyan birikiyor (a + a, a·a)\", reuse)\n\n    def deep():\n        def f(x, y):\n            z = x * y + x ** 3 - y / 2.5\n            q = (z.tanh() * 2 + 1) ** 2\n            return q - 3 * x + (1 - y) * z\n        def fv(x, y):\n            z = x * y + x ** 3 - y / 2.5\n            q = (math.tanh(z) * 2 + 1) ** 2\n            return q - 3 * x + (1 - y) * z\n        for x0, y0 in [(0.5, -0.3), (-0.8, 1.1), (0.1, 0.2)]:\n            x, y = V(x0), V(y0)\n            out = f(x, y)\n            out.backward()\n            h = 1e-6\n            nx = (fv(x0 + h, y0) - fv(x0 - h, y0)) / (2 * h)\n            ny = (fv(x0, y0 + h) - fv(x0, y0 - h)) / (2 * h)\n            close([out.data, x.grad, y.grad], [fv(x0, y0), nx, ny], rtol=1e-5, atol=1e-6, what=f\"({x0}, {y0})\")\n        return \"sayısal türevle 6 basamak eşleşti\"\n    R.test(\"uzun ifade (−, /, **, r-işlemleri) sayısal türevle eşleşiyor\", deep)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n\n    def hand():\n        o, L, dw, db = need(ns, \"neuron_fb\")(np.array([[2.0, 0.0]]), np.array([-3.0, 1.0]), 6.8813735870195432)\n        close(o, [2 ** -0.5]); close(dw, [1.0, 0.0], atol=1e-9); close(db, 0.5)\n    R.test(\"elle nöron: o = 1/√2, ∂w = [1, 0], ∂b = 0.5\", hand)\n\n    def rnd():\n        rng = np.random.default_rng(0)\n        X, w, b = rng.normal(size=(50, 4)), rng.normal(size=4), 0.3\n        o, L, dw, db = need(ns, \"neuron_fb\")(X, w, b)\n        h = 1e-6\n        f = lambda w_, b_: np.tanh(X @ w_ + b_).sum()\n        E = h * np.eye(4)\n        num_w = np.array([(f(w + E[j], b) - f(w - E[j], b)) / (2 * h) for j in range(4)])\n        num_b = (f(w, b + h) - f(w, b - h)) / (2 * h)\n        close(L, f(w, b), what=\"L\"); close(dw, num_w, rtol=1e-6, atol=1e-6, what=\"dw\"); close(db, num_b, rtol=1e-6, atol=1e-6, what=\"db\")\n        ensure(np.asarray(o).shape == (50,), f\"o shape {np.asarray(o).shape}\")\n    R.test(\"50 örnek: sayısal gradyanla eşleşiyor\", rnd)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    def kg():\n        g = need(ns, \"karpathy_grads\")()\n        ensure(isinstance(g, dict) and set(g) >= {\"a\", \"b\", \"c\", \"f\"}, \"{'a','b','c','f'} anahtarlı sözlük bekleniyordu\")\n        close([float(g[k]) for k in \"abcf\"], [6, -4, -2, 4])\n    R.test(\"karpathy_grads: elle bulduğunla aynı\", kg)\n\n    def nt():\n        out = need(ns, \"neuron_torch\")(torch.tensor([2.0, 0.0], dtype=torch.float64), torch.tensor([-3.0, 1.0], dtype=torch.float64), 6.8813735870195432)\n        ensure(isinstance(out, (tuple, list)) and len(out) == 3, \"(o, dw, dx) döndürmeli\")\n        o, dw, dx = out\n        close(o, 2 ** -0.5); close(dw, [1.0, 0.0], atol=1e-9); close(dx, [-1.5, 0.5])\n    R.test(\"neuron_torch: ∂w = [1, 0], ∂x = [−1.5, 0.5]\", nt)\n\n    def graph():\n        a = torch.tensor(2.0, requires_grad=True); b = torch.tensor(-3.0, requires_grad=True)\n        L = (a * b + 10) * -2\n        names, fn = [], L.grad_fn\n        while fn is not None:\n            names.append(type(fn).__name__)\n            nxt = [f for f, _ in fn.next_functions if f is not None and type(f).__name__ != \"AccumulateGrad\"]\n            fn = nxt[0] if nxt else None\n        return \"PyTorch'un grafiği (çıktıdan geriye): \" + \" → \".join(names)\n    R.test(\"PyTorch'un hesap grafiği\", graph)\n",
 "e": "import torch\nfrom labcheck import close, no_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    g = torch.Generator().manual_seed(0)\n    X, w, b = torch.randn(64, 5, generator=g, dtype=torch.float64), torch.randn(5, generator=g, dtype=torch.float64), 0.2\n\n    def ps():\n        G = need(ns, \"per_sample_grads\")(X, w, b)\n        ensure(tuple(G.shape) == (64, 5), f\"shape {tuple(G.shape)}, (64, 5) bekleniyordu\")\n        o = torch.tanh(X @ w + b)\n        close(G, (1 - o ** 2)[:, None] * X, what=\"gᵢ = (1 − oᵢ²) xᵢ\")\n        close(G.sum(0), X.T @ (1 - o ** 2), what=\"satır toplamı = batch gradyanı\")\n        return \"satırlar toplanınca c aşamasındaki X.T @ (1 − o²) çıkıyor\"\n    R.test(\"örnek başına gradyan doğru, toplamı batch gradyanı\", ps)\n    if \"per_sample_grads\" not in ns:\n        return\n    Xb, wb = torch.randn(1024, 32), torch.randn(32)\n\n    def loop():\n        out = []\n        for x in Xb:\n            ww = wb.clone().requires_grad_(True)\n            torch.tanh(x @ ww + 0.1).backward()\n            out.append(ww.grad)\n        return torch.stack(out)\n    t_loop = bench(loop, repeat=3, number=1)\n    t_vm = bench(lambda: ns[\"per_sample_grads\"](Xb, wb, 0.1), repeat=3)\n    R.table(\"1024 örnek, d = 32: örnek başına gradyan\", [\"yöntem\", \"süre\", \"döngüye göre\"],\n            [[\"for + backward (1024 grafik)\", fmt_time(t_loop), \"1×\"],\n             [\"senin per_sample_grads (vmap)\", fmt_time(t_vm), f\"{t_loop / t_vm:,.0f}× hızlı\"]],\n            note=\"Skaler micrograd ile aynı hesap 1024 × 32 Value düğümü demek; vmap birkaç tensör işlemine indirir.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m3_1_micrograd as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

`loss.backward()` yazdığında PyTorch ne yapıyor? Bu modülde cevabı kendin yazıyorsun. Fikir Karpathy'nin micrograd'ından: her sayı bir `Value` nesnesine sarılır ve üç şeyi hatırlar:

1. `data` — değeri,
2. `_prev` — kendisini üreten girdileri (grafiğin kenarları),
3. `_backward` — kendi yerel türevini girdilerine nasıl dağıtacağını bilen küçük bir fonksiyon.

`L.backward()` grafiği topolojik olarak sıralar, `L.grad = 1` yapar ve düğümleri **ters** sırada gezip her birinin `_backward`'ını çağırır. M0.3'te elle yaptığın her şey, ama otomatik.

Kullanacağımız yerel türevler:

| işlem | ileri | geri (gelen gradyan $g$ ile) |
|---|---|---|
| $o=a+b$ | | $a \mathrel{+}= g$, $b \mathrel{+}= g$ |
| $o=a\cdot b$ | | $a \mathrel{+}= g\cdot b$, $b \mathrel{+}= g\cdot a$ |
| $o=a^k$ | | $a \mathrel{+}= g\cdot k\,a^{k-1}$ |
| $o=\tanh a$ | | $a \mathrel{+}= g\,(1-o^2)$ |

## 1 · Elle

Grafiği çiz: $a$ ve $b$ bir çarpma düğümüne, sonucu $e$ ile $c$ bir toplama düğümüne, sonucu $d$ ile $f$ son çarpma düğümüne giriyor.

Geri geçişte iki kuralı uygula: **toplama** düğümü gelen gradyanı değiştirmeden dağıtır; **çarpma** düğümü her girdiye "gelen gradyan × diğer girdinin değeri" verir. Bu iki kural, sinir ağlarındaki gradyan akışının sezgisinin yarısıdır.

İkinci grupta tek bir tanh nöronu var. $w_2$'nin gradyanının neden 0 olduğunu düşün: girdisi $x_2=0$ olan bir ağırlık o örnekten hiçbir şey öğrenemez.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Grafik: $e=a\cdot b$, $d=e+c$, $L=d\cdot f$; $a=2$, $b=-3$, $c=10$, $f=-2$. İleri geçiş**

- `e`: $e$
- `d`: $d$
- `L`: $L$

**Geri geçiş ($\partial L/\partial L=1$ ile başla)**

- `gd`: $\partial L/\partial d$
- `gf`: $\partial L/\partial f$
- `gc`: $\partial L/\partial c$
- `ge`: $\partial L/\partial e$
- `ga`: $\partial L/\partial a$
- `gb`: $\partial L/\partial b$

**Nöron: $n=x_1w_1+x_2w_2+b$, $o=\tanh(n)$; $x=(2,0)$, $w=(-3,1)$, $b=6.8814$ (öyle seçildi ki $o=1/\sqrt2$)**

- `n`: $n$ (3 ondalık)
- `dodn`: $\partial o/\partial n=1-o^2$
- `dw1`: $\partial o/\partial w_1$
- `dw2`: $\partial o/\partial w_2$
- `dx1`: $\partial o/\partial x_1$

In [ ]:
cevaplar = dict(
    e=None,
    d=None,
    L=None,
    gd=None,
    gf=None,
    gc=None,
    ge=None,
    ga=None,
    gb=None,
    n=None,
    dodn=None,
    dw1=None,
    dw2=None,
    dx1=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"e":[-6,0],"d":[4,0],"L":[-8,0],"gd":[-2,0],"gf":[4,0],"gc":[-2,0],"ge":[-2,0],"ga":[6,0],"gb":[-4,0],"n":[0.8814,0.002],"dodn":[0.5,0.001],"dw1":[1,0.002],"dw2":[0,0.002],"dx1":[-1.5,0.002]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

`Value` sınıfını tamamla. `__init__`, `__repr__` ve `__add__` hazır; `__add__`'ın `_backward`'ını şablon olarak oku.

Yazılacaklar:

- `__mul__` — `other` sayıysa önce `Value`'ya sar.
- `__pow__(k)` — `k` yalnız `int`/`float`.
- `tanh()` — `math.tanh` serbest.
- `backward()` — derinlik öncelikli arama ile topolojik sıra kur (ziyaret edilenleri bir `set`'te tut), `self.grad = 1.0`, sonra ters sırada `_backward()` çağır.
- Kolaylıklar: `__radd__`, `__rmul__`, `__neg__`, `__sub__`, `__rsub__`, `__truediv__` (son ikisi `__pow__(-1)` ile).

Kontrol: elle örnek, tanh nöronu, `a + a` (gradyan birikmesi), uzun bir ifadenin sayısal türev ile karşılaştırılması.

In [ ]:
%%writefile stage_b.py
import math


class Value:
    def __init__(self, data, _prev=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._prev = set(_prev)
        self._op = _op
        self._backward = lambda: None

    def __repr__(self):
        return f"Value(data={self.data:.4g}, grad={self.grad:.4g})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        ...

    def __pow__(self, k):
        ...

    def tanh(self):
        ...

    def backward(self):
        ...

    # kolaylıklar
    def __radd__(self, other): ...
    def __rmul__(self, other): ...
    def __neg__(self): ...
    def __sub__(self, other): ...
    def __rsub__(self, other): ...
    def __truediv__(self, other): ...


a, b, c, f = Value(2), Value(-3), Value(10), Value(-2)
L = (a * b + c) * f
L.backward()
print(L, a.grad, b.grad, c.grad, f.grad)   # -8, 6, -4, -2, 4

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

Skaler motor her sayı için bir Python nesnesi oluşturur; $n$ örneklik bir batch için $n\cdot d$ düğüm. Tensör düzeyinde aynı nöronun ileri ve geri geçişi birkaç satır:

$$
o=\tanh(Xw+b)\in\mathbb{R}^{n},\qquad L=\sum_i o_i,\qquad
\nabla_w L=X^\top(1-o^2),\qquad \frac{\partial L}{\partial b}=\sum_i(1-o_i^2).
$$

- `neuron_fb(X, w, b)` → `(o, L, dw, db)`. **Döngü yok.** Bu, `Value` grafiğinde her örnek için ayrı ayrı yaptığın şeyin matris hâli: $X^\top$ çarpımı, örnekler üzerindeki gradyan toplamını tek işlemde yapıyor.

In [ ]:
%%writefile stage_c.py
import numpy as np


def neuron_fb(X, w, b):
    """o = tanh(Xw + b), L = Σo. → (o, L, dL/dw, dL/db). Döngü yok."""
    ...


X = np.array([[2.0, 0.0], [1.0, -1.0], [0.0, 0.5]])
w = np.array([-3.0, 1.0])
print(neuron_fb(X, w, 6.8813735870195432))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Şimdi aynı iki örneği PyTorch'a yaptır ve elle bulduklarınla karşılaştır.

- `karpathy_grads()` — $a=2$, $b=-3$, $c=10$, $f=-2$ için `requires_grad=True` skaler tensörlerle $L=(a\cdot b+c)\cdot f$; `{"a": …, "b": …, "c": …, "f": …}` sözlüğüyle gradyanları döndür.
- `neuron_torch(x, w, b)` → `(o, dw, dx)` — $o=\tanh(x\cdot w+b)$'nin $w$ ve $x$'e göre gradyanları. (Girdiye göre gradyan, adversarial örnekler ve saliency haritalarının temeli.)

Kontrol ayrıca `o.grad_fn` zincirini gezerek PyTorch'un grafiğinin seninkine benzediğini gösterir.

In [ ]:
%%writefile stage_d.py
import torch


def karpathy_grads():
    """L = (a·b + c)·f için {'a','b','c','f'} → gradyan."""
    ...


def neuron_torch(x, w, b):
    """o = tanh(x·w + b). → (o, ∂o/∂w, ∂o/∂x)"""
    x = x.detach().clone().requires_grad_(True)
    w = w.detach().clone().requires_grad_(True)
    ...


print(karpathy_grads())
print(neuron_torch(torch.tensor([2.0, 0.0]), torch.tensor([-3.0, 1.0]), 6.8813735870195432))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`loss.backward()` batch kaybının **toplam** gradyanını verir. Bazen her örneğin ayrı gradyanı gerekir: differential privacy (örnek başına kırpma), veri değerlendirme (hangi örnek modeli ne kadar itiyor), meta-öğrenme.

Döngüyle $n$ kez backward ya da `torch.func`:

```python
from torch.func import grad, vmap
def loss_one(w, b, x):          # tek örnek
    return torch.tanh(x @ w + b)
vmap(grad(loss_one), in_dims=(None, None, 0))(w, b, X)   # (n, d)
```

`in_dims` hangi argümanın batch'lendiğini söyler: `w` ve `b` paylaşılıyor (`None`), `X` 0. eksende batch.

- `per_sample_grads(X, w, b)` → `(n, d)`; satır $i$: $\partial\tanh(x_i\cdot w+b)/\partial w$. **Döngü yok.**

In [ ]:
%%writefile stage_e.py
import torch
from torch.func import grad, vmap


def loss_one(w, b, x):
    return torch.tanh(x @ w + b)


def per_sample_grads(X, w, b):
    """→ (n, d): her satır ∂tanh(xᵢ·w + b)/∂w. Döngü yok."""
    ...


X = torch.randn(4, 3)
w = torch.randn(3)
print(per_sample_grads(X, w, 0.1))

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Paszke ve ark. (2017): PyTorch'ta otomatik türev

[Paszke ve ark. — Automatic differentiation in PyTorch (NIPS 2017 Autodiff Workshop)](https://openreview.net/forum?id=BJJsrmfCZ)

micrograd'ını makalenin tasarım ilkelerine göre genişlet: (1) dinamik, çalıştırarak tanımlanan (define-by-run) grafik, (2) yerinde (in-place) işlemleri sürüm sayaçlarıyla güvenli kılma (Bölüm 3), (3) create_graph ile ikinci türevler (grafiğin kendisini türevlenebilir yapmak). Sürüm sayacının yakaladığı bir 'yerinde değişiklik' hatası örneği üret; ikinci türevleri sayısal türevlerle doğrula.


**Adımlar**

1. Value'ya _version alanı ekle; geri geçişte kaydedilen sürümle karşılaştır.

**Kabul ölçütleri**

- Yerinde değişiklik geri geçişte yakalanır.
- Hessian-vektör çarpımı sonlu farkla eşleşir.

## Kaynaklar

- [Karpathy — micrograd](https://github.com/karpathy/micrograd) — MIT
- [Karpathy — Neural Networks: Zero to Hero, 1: The spelled-out intro to neural networks and backpropagation](https://karpathy.ai/zero-to-hero.html) — kod MIT; video ©
- [CS231n — Backpropagation, Intuitions](https://cs231n.github.io/optimization-2/) — MIT
- [MiniTorch — Module 1: Autodiff](https://minitorch.github.io/) — lisans dosyası yok
- [PyTorch — torch.func: per-sample gradients](https://docs.pytorch.org/tutorials/intermediate/per_sample_grads.html) — BSD-3